# ✈️ Airline Behavioral Intelligence — Full Pipeline (Google Colab)

## ⚡ ENABLE GPU FIRST (takes 30 seconds, saves 5+ minutes on training)
> **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

Then click **Runtime → Run all**

---

### What this notebook does:
| Step | Action |
|------|--------|
| 1 | Install dependencies + check GPU |
| 2 | Clone project from GitHub |
| 3 | Upload your 2 data CSV files |
| 4 | Run Stages 2–7 (cleaning → retention engine) |
| 5 | Display model AUC, segment breakdown, revenue at risk |
| 6 | Download all outputs (models, figures, predictions) |

### Required data files (upload when prompted):
- `Customer Loyalty History.csv`
- `Customer Flight Activity.csv`

---
## ⚙️ SECTION 1 — Setup

In [ ]:
# ── Check GPU availability ────────────────────────────────────────────────────
import subprocess, sys

try:
    gpu_info = subprocess.check_output('nvidia-smi', stderr=subprocess.DEVNULL).decode()
    # Extract GPU name
    for line in gpu_info.split('\n'):
        if 'Tesla' in line or 'T4' in line or 'A100' in line or 'V100' in line or 'L4' in line:
            print(f'🚀 GPU detected: {line.strip()}')
            break
    else:
        print('🚀 GPU detected (nvidia-smi found)')
    GPU_AVAILABLE = True
    print('   XGBoost will train on CUDA — expect ~10x speedup')
except Exception:
    GPU_AVAILABLE = False
    print('💻 No GPU detected — running on CPU')
    print('   To enable GPU: Runtime → Change runtime type → T4 GPU')
    print('   CPU is fine for this dataset (16k rows) — will take ~5 min')

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
print('Installing packages... (60-90 seconds)\n')

# Use XGBoost 2.0+ for clean GPU support (device='cuda')
packages = [
    'pandas==2.0.3',
    'numpy==1.24.3',
    'scikit-learn==1.3.0',
    'xgboost>=2.0.0',
    'imbalanced-learn==0.11.0',
    'matplotlib==3.7.2',
    'seaborn==0.12.2',
    'plotly==5.15.0',
    'joblib==1.3.1',
    'shap',
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✓' if result.returncode == 0 else '✗'
    print(f'  {status} {pkg}')

print('\n✅ All packages installed.')

In [ ]:
# ── Clone project from GitHub ─────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/Arjun-Gangwar1/Airline-Loyalty-Programs.git'
PROJ_DIR = '/content/airline_project'

if os.path.exists(PROJ_DIR):
    print('Project already cloned. Pulling latest...')
    os.system(f'cd {PROJ_DIR} && git pull -q')
else:
    print('Cloning from GitHub...')
    result = os.system(f'git clone -q {REPO_URL} {PROJ_DIR}')
    print('✅ Cloned successfully' if result == 0 else '❌ Clone failed')

os.chdir(PROJ_DIR)
if PROJ_DIR not in sys.path:
    sys.path.insert(0, PROJ_DIR)

print(f'Working directory: {os.getcwd()}')
print('Files:', [f for f in os.listdir('.') if not f.startswith('.')])

In [ ]:
# ── Create directory structure ────────────────────────────────────────────────
from pathlib import Path
import json

dirs = [
    'data/raw', 'data/processed', 'data/final',
    'outputs/models',
    'outputs/figures/exploration', 'outputs/figures/churn',
    'outputs/figures/features',    'outputs/figures/segmentation',
    'outputs/figures/models',      'outputs/figures/retention',
    'outputs/reports', 'checkpoints', 'logs',
]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

progress = {'current_stage': 0, 'completed_stages': [], 'last_updated': ''}
with open('checkpoints/progress.json', 'w') as f:
    json.dump(progress, f, indent=2)

print('✅ Directory structure ready.')

In [ ]:
# ── Upload data files ─────────────────────────────────────────────────────────
# Upload BOTH CSV files when the dialog appears:
#   • Customer Loyalty History.csv
#   • Customer Flight Activity.csv
from google.colab import files

print('📂 Select BOTH CSV files (hold Ctrl/Cmd to select multiple)...')
uploaded = files.upload()

for filename, content in uploaded.items():
    dest = Path('data/raw') / filename
    dest.write_bytes(content)
    print(f'  ✓ {filename}  ({len(content)/1024:.0f} KB)')

raw_files = list(Path('data/raw').glob('*.csv'))
print(f'\n✅ {len(raw_files)} file(s) ready in data/raw/')

if len(raw_files) < 2:
    print('⚠️  Only 1 file uploaded — please re-run this cell and upload BOTH files.')

---
## 🔄 SECTION 2 — Run the Pipeline

Each stage runs as a subprocess and prints its full output. If a stage fails, fix the error and re-run that single cell.

In [ ]:
# ── Stage runner helper ───────────────────────────────────────────────────────
import time

def run_stage(script, num, label):
    print(f'\n{"="*60}')
    print(f'  STAGE {num}: {label}')
    print(f'{"="*60}')
    t0 = time.time()
    r  = subprocess.run(
        [sys.executable, script],
        capture_output=True, text=True, cwd=PROJ_DIR
    )
    elapsed = time.time() - t0
    print(r.stdout[-4000:] if len(r.stdout) > 4000 else r.stdout)
    if r.returncode != 0:
        print('STDERR:\n', r.stderr[-2000:])
        raise RuntimeError(f'Stage {num} failed — see output above.')
    print(f'  ✅ Done in {elapsed:.1f}s')

In [ ]:
run_stage('2_data_cleaning.py', 2, 'DATA CLEANING')

In [ ]:
run_stage('3_churn_definition.py', 3, 'CHURN DEFINITION')

In [ ]:
run_stage('4_feature_engineering.py', 4, 'FEATURE ENGINEERING')

In [ ]:
run_stage('5_customer_segmentation.py', 5, 'CUSTOMER SEGMENTATION')

In [ ]:
# Stage 6 auto-detects GPU — if GPU is enabled you will see:
# "🚀 GPU detected — training on CUDA"
run_stage('6_baseline_models.py', 6, 'CHURN PREDICTION MODELS')

In [ ]:
run_stage('7_retention_engine.py', 7, 'RETENTION ENGINE')

---
## 📊 SECTION 3 — Results

In [ ]:
# ── Model performance ─────────────────────────────────────────────────────────
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')

reports = Path('outputs/reports')
final   = Path('data/final')

print('='*60)
print('  MODEL PERFORMANCE')
print('='*60)

mc_path = reports / 'model_comparison.csv'
if mc_path.exists():
    mc = pd.read_csv(mc_path)
    show = [c for c in ['Model','ROC_AUC','Recall','Precision','F1','CV_AUC_mean'] if c in mc.columns]
    print(mc[show].to_string(index=False))
    best = mc.loc[mc['ROC_AUC'].idxmax()]
    print(f'\n🏆 Best  : {best["Model"]}')
    print(f'   AUC   : {best["ROC_AUC"]:.4f}')
    print(f'   Recall: {best["Recall"]:.4f}  → catches {best["Recall"]*100:.0f}% of churners')
    print(f'   Prec  : {best["Precision"]:.4f}  → {best["Precision"]*100:.0f}% of alerts are real')
else:
    print('model_comparison.csv not found — re-run Stage 6.')

In [ ]:
# ── Churn predictions ─────────────────────────────────────────────────────────
print('='*60)
print('  CHURN PREDICTIONS')
print('='*60)

pp = final / 'all_customer_predictions.csv'
if not pp.exists(): pp = final / 'churn_predictions.csv'

if pp.exists():
    preds = pd.read_csv(pp)
    cc = 'predicted_churn' if 'predicted_churn' in preds.columns else 'churn'
    n  = len(preds)
    at = preds[cc].sum()
    print(f'  Total customers  : {n:,}')
    print(f'  Predicted churners: {at:,} ({at/n*100:.1f}%)')
    print(f'  Safe customers   : {n-at:,} ({(n-at)/n*100:.1f}%)')
    if 'risk_level' in preds.columns:
        print('\n  Risk breakdown:')
        for lv in ['high','medium','low']:
            k = (preds['risk_level']==lv).sum()
            print(f'    {lv.upper():<8}: {k:,}  ({k/n*100:.1f}%)')

In [ ]:
# ── Customer segments ─────────────────────────────────────────────────────────
print('='*60)
print('  CUSTOMER SEGMENTS')
print('='*60)

sp = final / 'customer_features_segmented.csv'
if not sp.exists(): sp = final / 'customer_segments.csv'

if sp.exists():
    seg = pd.read_csv(sp)
    sc  = 'segment_name' if 'segment_name' in seg.columns else 'segment'
    if sc in seg.columns:
        cc2  = next((c for c in ['churn','churned'] if c in seg.columns), None)
        clvc = 'clv' if 'clv' in seg.columns else None
        for sname, cnt in seg[sc].value_counts().items():
            mask = seg[sc] == sname
            extra = ''
            if cc2:  extra += f'  churn={seg.loc[mask,cc2].mean()*100:.0f}%'
            if clvc: extra += f'  CLV=${seg.loc[mask,clvc].median():,.0f}'
            print(f'  {sname:<28} {cnt:5,} ({cnt/len(seg)*100:.1f}%){extra}')

In [ ]:
# ── Revenue at risk ───────────────────────────────────────────────────────────
print('='*60)
print('  REVENUE IMPACT')
print('='*60)

rp = reports / 'retention_actions.csv'
if rp.exists():
    ret = pd.read_csv(rp)
    if 'revenue_at_risk' in ret.columns:
        tr = ret['revenue_at_risk'].sum()
        ts = ret['potential_save'].sum() if 'potential_save' in ret.columns else 0
        print(f'  Total revenue at risk : ${tr:,.0f}')
        print(f'  Potential savings     : ${ts:,.0f}')
        if 'segment_name' in ret.columns:
            print('\n  By segment:')
            for s, v in ret.groupby('segment_name')['revenue_at_risk'].sum().sort_values(ascending=False).items():
                print(f'    {s:<28} ${v:>10,.0f}')

In [ ]:
# ── Display all figures inline ────────────────────────────────────────────────
from IPython.display import display, Image
import glob, matplotlib.pyplot as plt

all_figs = sorted(glob.glob('outputs/figures/**/*.png', recursive=True))
print(f'{len(all_figs)} figures generated:\n')
for p in all_figs:
    print(f'  • {p}')

# Show the most important ones
priority = ['roc_curves', 'feature_importance', 'segment_overview',
            'segments_pca', 'revenue_at_risk', 'feature_correlations']

print('\n--- KEY FIGURES ---')
for kw in priority:
    matches = [p for p in all_figs if kw in p]
    if matches:
        print(f'\n📊 {matches[0]}')
        display(Image(filename=matches[0], width=880))

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────────
import joblib

model_pkl = Path('outputs/models/best_model.pkl')
feat_csv  = final / 'customer_features_segmented.csv'
if not feat_csv.exists(): feat_csv = final / 'customer_features.csv'

if model_pkl.exists() and feat_csv.exists():
    model   = joblib.load(model_pkl)
    feat_df = pd.read_csv(feat_csv)
    skip    = {'loyalty_number','churn','churned','hard_churn','activity_churn',
               'activity_churn_12m','activity_churn_6m','segment_name','segment',
               'cluster','gender','education','marital_status','loyalty_card',
               'enrollment_type','country','province','city','postal_code',
               'last_activity_date','cancellation_year','cancellation_month'}
    fcols   = [c for c in feat_df.columns if c not in skip and feat_df[c].dtype != object]

    if hasattr(model, 'feature_importances_') and len(model.feature_importances_) == len(fcols):
        imp = pd.Series(model.feature_importances_, index=fcols).nlargest(15)
        print('Top 15 Features (from best model):')
        print('='*48)
        for feat, val in imp.items():
            bar = '█' * int(val * 400)
            print(f'  {feat:<35} {val:.4f}  {bar}')
        imp_df = imp.reset_index()
        imp_df.columns = ['feature','importance']
        imp_df.to_csv(reports / 'feature_importance.csv', index=False)
        print('\n✓ Saved: outputs/reports/feature_importance.csv')
else:
    print('Model or feature file not found.')

In [ ]:
# ── Pipeline summary JSON ─────────────────────────────────────────────────────
from datetime import datetime

summary = {
    'run_date':     datetime.now().isoformat(),
    'gpu_used':     GPU_AVAILABLE,
    'status':       'complete',
}

if (reports / 'model_comparison.csv').exists():
    mc = pd.read_csv(reports / 'model_comparison.csv')
    for _, row in mc.iterrows():
        name = row.get('Model', row.get('model','unknown'))
        summary[f'{name.replace(" ","_")}_auc'] = float(row.get('ROC_AUC', row.get('roc_auc', 0)))

pp2 = final / 'all_customer_predictions.csv'
if pp2.exists():
    p2 = pd.read_csv(pp2)
    cc3 = 'predicted_churn' if 'predicted_churn' in p2.columns else 'churn'
    summary['total_customers']    = int(len(p2))
    summary['predicted_churners'] = int(p2[cc3].sum())
    summary['churn_rate_pct']     = round(float(p2[cc3].mean() * 100), 2)

if (reports / 'retention_actions.csv').exists():
    r2 = pd.read_csv(reports / 'retention_actions.csv')
    if 'revenue_at_risk' in r2.columns:
        summary['total_revenue_at_risk'] = round(float(r2['revenue_at_risk'].sum()), 2)
    if 'potential_save' in r2.columns:
        summary['total_potential_save']  = round(float(r2['potential_save'].sum()), 2)

with open(reports / 'pipeline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print('\n✅ Saved: outputs/reports/pipeline_summary.json')

---
## 📦 SECTION 4 — Download Outputs

In [ ]:
# ── Zip everything and download ───────────────────────────────────────────────
import shutil
from google.colab import files

results_dir = Path('/content/airline_results')
if results_dir.exists():
    shutil.rmtree(results_dir)
results_dir.mkdir()

for src, dst in [
    ('outputs',    results_dir / 'outputs'),
    ('data/final', results_dir / 'data_final'),
]:
    if Path(src).exists():
        shutil.copytree(src, dst)
        print(f'✓ Packed {src}')

zip_path = '/content/airline_outputs'
shutil.make_archive(zip_path, 'zip', results_dir)
size_mb  = Path(zip_path + '.zip').stat().st_size / 1024 / 1024
print(f'\n✅ airline_outputs.zip ready ({size_mb:.1f} MB)')
files.download(zip_path + '.zip')

In [ ]:
# ── [Optional] Save to Google Drive ──────────────────────────────────────────
# Uncomment below if you prefer Drive over download

# from google.colab import drive
# drive.mount('/content/drive')
# drive_dest = '/content/drive/MyDrive/AirlineProject'
# if Path(drive_dest).exists(): shutil.rmtree(drive_dest)
# shutil.copytree(str(results_dir), drive_dest)
# print(f'✅ Saved to Google Drive: {drive_dest}')

# ── [Optional] Download individual files ─────────────────────────────────────
# files.download('outputs/models/best_model.pkl')
# files.download('outputs/models/xgboost.pkl')
# files.download('data/final/all_customer_predictions.csv')
# files.download('outputs/reports/retention_actions.csv')
# files.download('outputs/reports/model_comparison.csv')
# files.download('outputs/reports/pipeline_summary.json')
# files.download('outputs/figures/models/roc_curves.png')
# files.download('outputs/figures/models/feature_importance.png')

print('Uncomment any line above for Drive save or individual file download.')

---
## ✅ Done!

### Your `airline_outputs.zip` contains:
```
outputs/
  models/
    best_model.pkl          ← best churn model (XGBoost)
    xgboost.pkl
    random_forest.pkl
    logistic_regression.pkl
    feature_scaler.pkl
  figures/
    models/roc_curves.png
    models/feature_importance.png
    segmentation/segments_pca.png
    retention/revenue_at_risk.png
  reports/
    model_comparison.csv
    retention_actions.csv
    priority_action_list.csv
    feature_importance.csv
    pipeline_summary.json   ← copy AUC numbers from here to your README
    retention_playbook.json
data_final/
  customer_features.csv
  customer_features_segmented.csv
  all_customer_predictions.csv
  churn_predictions.csv
  customer_segments.csv
```

**After downloading:** open `pipeline_summary.json` and update your README with the real AUC score.